# Tier-0 · 0.1 — CTC baseline on AraMS-28k-HTR (crash-safe)

Two line-recognition baselines on the AraMS-28k-HTR **book-level** split:

1. **Kraken** — `ketos train` warm-started from the Muharaf recognition model
   (Zenodo `10.5281/zenodo.14295489`). The citable baseline.
2. **Own CRNN** — CNN → 2×BiLSTM(256) → CTC, height 64, AdamW 3e-4 cosine,
   batch 32, grad-clip 5.0, early stop on val-CER (patience 10). The one you
   ablate.

**Sanity anchors (RefLAM, Kraken on the AraMS-28k test split):** weighted CER
**23.31 %** — Maghrebi 32.71 / Naskh 22.62 / Ruq'ah 11.65. Wildly off ⇒ the
bug is in the crops or the charset, not the model.

**Crash-safe.** Everything durable lives on Google Drive; the CRNN checkpoints
every `SAVE_EVERY` steps *and* every epoch, Kraken every epoch (`--savefreq 1`).
Reopen + **Run all** after a disconnect and both resume from the last
checkpoint. Per-eval metrics are appended to `metrics.jsonl` so nothing is
lost even if the final cell never runs.

**Before running:** `Runtime → T4 GPU`. Add **both** Colab secrets and tick
"Notebook access" for each (left sidebar → 🔑):
`GH_TOKEN` — a fine-grained PAT with *Contents: read* on
`rabeeh97/arabic-ocr-benchmark` (**required** — the repo is private and cell 3
clones it for `metrics.py` / `normalize.py`; also used to push the results
JSON), and `HF_TOKEN` — for the AraMS dataset. Run
`07_colab_mirror_arams.ipynb` first (or let cell 4 stream the mirror).

## 1. GPU

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Drive — the durable store (checkpoints + resume live here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, time
WORK = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/train/ctc-baseline')
(WORK / 'kraken').mkdir(parents=True, exist_ok=True)
(WORK / 'crnn').mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/arabic-ocr-bench/hf_cache'
print('durable dir:', WORK)

## 3. Repo (for metrics/normalize) + auth + install

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
# Copy Colab secrets into the environment. SWEEP_RUNNER_ID pins a STABLE
# name for this account (e.g. 'runner-a') so a restarted runner re-adopts
# its own in-flight batches instantly instead of waiting out the lease TTL.
for _k in ('GH_TOKEN', 'HF_TOKEN', 'SWEEP_RUNNER_ID'):
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v: os.environ[_k] = _v
    except Exception:
        pass
token = os.environ.get('GH_TOKEN')
if not token:
    print('no GH_TOKEN secret — clone/push will be anonymous / read-only')
import subprocess, time, sys
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
def _sh(*a):
    return subprocess.run(a, capture_output=True, text=True)
_updated = False
if os.path.isdir(REPO + '/.git'):
    r = _sh('git', '-C', REPO, 'fetch', '--depth', '1', 'origin', 'main')
    if r.returncode == 0:
        _sh('git', '-C', REPO, 'reset', '--hard', 'origin/main')
        # drop untracked leftovers (e.g. a stale sweep_manifest.json); keeps gitignored _cache
        _sh('git', '-C', REPO, 'clean', '-fd', '--', 'results')
        _updated = True
    else:
        print('existing checkout is stale and fetch failed — re-cloning\n', r.stderr[-300:])
        _sh('rm', '-rf', REPO)
if not _updated:
    for _try in range(1, 5):
        _sh('rm', '-rf', REPO)
        r = _sh('git', 'clone', '--depth', '1', url, REPO)
        if r.returncode == 0 and os.path.isdir(REPO + '/.git'):
            break
        print(f'clone attempt {_try}/4 failed:\n' + (r.stderr[-400:] or r.stdout[-400:]))
        time.sleep(5 * _try)
if not os.path.isdir(REPO + '/.git'):
    raise RuntimeError(
        'git clone failed. Checklist: (1) add a Colab secret named GH_TOKEN — a '
        'fine-grained PAT with Contents: read on rabeeh97/arabic-ocr-benchmark '
        '(the repo is private); (2) enable notebook access to that secret in the '
        'left sidebar key icon; (3) if it is a network blip, just re-run this cell.'
    )
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(_sh('git', 'log', '--oneline', '-3').stdout)

In [ ]:
import os, sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
from google.colab import userdata
for k in ('HF_TOKEN', 'GH_TOKEN'):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception as e:
        print('no', k, 'secret:', e)
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
# the repo's metrics.py needs jiwer; everything else the CRNN uses
# (torch, PIL, datasets) is already on Colab. kraken is installed in cell 6
# only if RUN includes it (it re-pins torch, so keep it out of the way).
!pip install -q jiwer
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. Materialise the data as an `images/` tree + manifests

Prefers the Kraken tar `07_colab_mirror_arams.ipynb` wrote to Drive; falls
back to streaming `<hf-user>/arams28k-htr-lines` and rebuilding the tree.

In [ ]:
import pathlib, tarfile, csv
DATA = pathlib.Path('/content/arams_htr')
TAR = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/data/arams28k-htr-kraken.tar')
IMAGES = DATA / 'images'
if not IMAGES.exists():
    DATA.mkdir(parents=True, exist_ok=True)
    if TAR.exists():
        print('restoring from', TAR)
        with tarfile.open(TAR) as tf: tf.extractall(DATA)
    else:
        print('no Drive tar — rebuilding from the HF mirror')
        from datasets import load_dataset
        from huggingface_hub import whoami
        repo = f"{whoami()['name']}/arams28k-htr-lines"
        IMAGES.mkdir(parents=True, exist_ok=True)
        man = {s: open(DATA / f'{s}_manifest.txt', 'w') for s in ('train','val','test')}
        meta_rows = []
        for split in ('train', 'val', 'test'):
            for r in load_dataset(repo, split=split):
                sid = f"{r['book_id']}_{r['page_id']}_{r['line_idx']}"
                r['image'].save(IMAGES / f'{sid}.png')
                (IMAGES / f'{sid}.gt.txt').write_text(r['text'], encoding='utf-8')
                man[split].write(sid + '\n')
                meta_rows.append({'sample_id': sid, 'script': r.get('script',''),
                                  'zone': r.get('zone','main'), 'split': split})
        for f in man.values(): f.close()
        with open(DATA / 'metadata.csv', 'w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=['sample_id','script','zone','split']); w.writeheader()
            w.writerows(meta_rows)
def _mid(s):
    # a manifest line may be a bare id, a filename, or a path — reduce to the
    # bare '{book}_{page}_{line}' stem that names the .png / .gt.txt pair
    s = pathlib.Path(s.strip()).name
    for suf in ('.gt.txt', '.png', '.jpg', '.jpeg', '.txt'):
        if s.endswith(suf): s = s[:-len(suf)]
    return s
SPLIT = {s: [_mid(l) for l in (DATA / f'{s}_manifest.txt').read_text().splitlines() if l.strip()]
         for s in ('train', 'val', 'test')}
# keep only ids that actually have both an image and a transcription
SPLIT = {s: [i for i in v if (IMAGES / f'{i}.png').exists() and (IMAGES / f'{i}.gt.txt').exists()]
         for s, v in SPLIT.items()}
assert sum(len(v) for v in SPLIT.values()) > 0, (
    'no image/gt pairs matched the manifests — check the extracted layout: '
    + str(sorted(p.name for p in DATA.iterdir())))
SCRIPT = {}
with open(DATA / 'metadata.csv', newline='') as fh:
    for row in csv.DictReader(fh):
        k = next((row[c] for c in row if c.lower() in ('sample_id','id','filename','name','image')), None)
        if k: SCRIPT[_mid(str(k))] = (row.get('script') or row.get('style') or '').lower()
print({s: len(v) for s, v in SPLIT.items()}, '| scripts:', sorted(set(SCRIPT.values()))[:6])

## 5. Config

`RUN` picks which baseline(s) to train. Checkpoint cadence is
**device-aware**: on a CPU runtime every step is far slower but a Colab
recycle is just as likely, so the CRNN checkpoints every 25 steps (vs 200
on GPU) and Kraken saves 5× per epoch (vs 1×). Override `SAVE_EVERY` /
`KRAKEN_SAVEFREQ` below if you want.

In [ ]:
import torch, random, numpy as np
RUN = 'crnn'          # 'crnn' | 'kraken' | 'both' — start with the self-contained
                      # CRNN; switch to 'both' once it trains and you want the
                      # citable Kraken baseline too.
HEIGHT = 64           # line-image height; 48 is the other arm
MAX_EPOCHS = 60
BATCH = 32
LR = 3e-4
GRAD_CLIP = 5.0
PATIENCE = 10         # early stop on val CER
EVAL_EVERY = 1        # epochs between val evals
SEED = 0

_ON_GPU = torch.cuda.is_available()
SAVE_EVERY = 200 if _ON_GPU else 25        # CRNN: optimiser steps between Drive checkpoints
KRAKEN_SAVEFREQ = 1.0 if _ON_GPU else 0.2  # Kraken: models saved per epoch (fraction ok)
print(('GPU' if _ON_GPU else 'CPU'), 'run -> SAVE_EVERY =', SAVE_EVERY,
      '| KRAKEN_SAVEFREQ =', KRAKEN_SAVEFREQ)
if not _ON_GPU:
    print('WARNING: no GPU — training will be very slow. Runtime -> Change runtime type -> T4 GPU.')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 6. Kraken baseline

Warm-starts from the Muharaf recognition model, or from this run's own
latest checkpoint on Drive if one exists (disconnect resume). `--savefreq 1`
writes a model to Drive every epoch; `--lag {PATIENCE}` is Kraken's early stop.

In [ ]:
import glob, subprocess, pathlib, shutil, urllib.request, json as _j
if RUN in ('kraken', 'both'):
    if shutil.which('ketos') is None:
        print('installing kraken (re-pins torch; only done for the Kraken arm)...')
        subprocess.run(['pip', 'install', '-q', 'kraken'])
    print('kraken:', subprocess.run(['ketos', '--version'], capture_output=True, text=True).stdout.strip())
    KOUT = WORK / 'kraken' / 'arams_ft'
    warm = sorted(glob.glob(str(WORK / 'kraken' / 'arams_ft_*.mlmodel')),
                  key=lambda p: pathlib.Path(p).stat().st_mtime)
    if warm:
        seed_model = warm[-1]; print('RESUME kraken from', seed_model)
    else:
        seed_model = str(WORK / 'kraken' / 'muharaf_rec.mlmodel')
        if not pathlib.Path(seed_model).exists():
            print('fetching Muharaf recognition model (Zenodo 14295489)...')
            rec = _j.loads(urllib.request.urlopen('https://zenodo.org/api/records/14295489').read())
            f = next((x for x in rec['files'] if x['key'].endswith('.mlmodel')), rec['files'][0])
            urllib.request.urlretrieve(f['links']['self'], seed_model)
            print('  got', f['key'])
        print('warm start from Muharaf model')
    (WORK / 'kraken' / 'train.files').write_text(
        '\n'.join(str(IMAGES / f'{i}.png') for i in SPLIT['train']))
    (WORK / 'kraken' / 'val.files').write_text(
        '\n'.join(str(IMAGES / f'{i}.png') for i in SPLIT['val']))
    _kdev = 'cuda:0' if _ON_GPU else 'cpu'
    cmd = ['ketos', 'train', '-f', 'path', '-i', seed_model, '--resize', 'add',
           '-o', str(KOUT), '-d', _kdev, '--savefreq', str(KRAKEN_SAVEFREQ),
           '--min-epochs', '30', '--lag', str(PATIENCE), '--augment',
           '-t', str(WORK / 'kraken' / 'train.files'),
           '-e', str(WORK / 'kraken' / 'val.files')]
    print(' '.join(cmd))
    r = subprocess.run(cmd, check=False, capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-3000:])
    if r.returncode != 0:
        print('\n!!! ketos train exited non-zero — read the error above. '
              'Flag names differ by kraken version (see `ketos train --help`), '
              'or set RUN="crnn" to skip Kraken entirely.')
else:
    print('RUN excludes kraken — skipping')

## 6b. Kraken eval — weighted + per-script CER vs RefLAM

In [ ]:
import glob, subprocess, re, pathlib, json
if RUN in ('kraken', 'both'):
    _cands = (sorted(glob.glob(str(WORK / 'kraken' / 'arams_ft_best.mlmodel'))) or
              sorted(glob.glob(str(WORK / 'kraken' / 'arams_ft_*.mlmodel')),
                     key=lambda p: pathlib.Path(p).stat().st_mtime))
    if not _cands:
        print('no Kraken model in', WORK / 'kraken', '— cell 6 (ketos train) produced none.')
        print('Read cell 6 output for the error, or set RUN="crnn" and skip to cell 7.')
    else:
        best = _cands[-1]; print('eval model:', best)
        def _ktest(ids):
            lf = WORK / 'kraken' / '_evalset.files'
            lf.write_text('\n'.join(str(IMAGES / f'{i}.png') for i in ids))
            out = subprocess.run(['ketos', 'test', '-m', best, '-f', 'path',
                                  '-t', str(lf), '-d', 'cuda:0' if _ON_GPU else 'cpu'],
                                 capture_output=True, text=True).stdout
            m = re.search(r'([Cc]haracter [Aa]ccuracy|Accuracy)[^0-9]*([0-9.]+)%', out)
            return (100 - float(m.group(2))) if m else None
        kr = {'weighted': _ktest(SPLIT['test'])}
        for sc in sorted({SCRIPT.get(i, '') for i in SPLIT['test']} - {''}):
            kr[sc] = _ktest([i for i in SPLIT['test'] if SCRIPT.get(i) == sc])
        print('Kraken CER %:', {k: (round(v, 2) if v is not None else None) for k, v in kr.items()})
        print('RefLAM      : weighted 23.31 | maghrebi 32.71 | naskh 22.62 | ruqah 11.65')
        (WORK / 'kraken' / 'result.json').write_text(json.dumps(kr, indent=2))

## 7. Own CRNN — self-contained, checkpoint-every-N-steps

Charset is the sorted set of characters in the **training** transcriptions
(index 0 = CTC blank). Line images are greyscale, height-normalised to
`HEIGHT` preserving aspect, right-padded per batch. The conv stack halves the
width twice (/4) and its height output is mean-pooled to a single row, so
the same code works for `HEIGHT` 64 or 48. This cell (re)builds the model and
loads `crnn/ckpt_last.pt` if a previous run left one.

In [ ]:
import pathlib, json, time, random
import torch, torch.nn as nn, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from arabic_ocr_bench.metrics import cer as _cer
from arabic_ocr_bench.normalize import normalize as _norm

CK = WORK / 'crnn' / 'ckpt_last.pt'
MET = WORK / 'crnn' / 'metrics.jsonl'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def _txt(sid):
    return (IMAGES / f'{sid}.gt.txt').read_text(encoding='utf-8').strip()

if CK.exists():
    _st = torch.load(CK, map_location='cpu')
    CHARSET = _st['charset']
    print('RESUME crnn @ step', _st.get('step', 0), '| charset', len(CHARSET))
else:
    CHARSET = ['<blank>'] + sorted({c for i in SPLIT['train'] for c in _txt(i)})
    _st = None; print('fresh crnn | charset', len(CHARSET))
STOI = {c: i for i, c in enumerate(CHARSET)}
OOV = sum(c not in STOI for i in SPLIT['train'] for c in _txt(i))
print('train chars:', len(CHARSET) - 1, '| OOV char occurrences vs this charset:', OOV)

class Lines(Dataset):
    def __init__(self, ids): self.ids = list(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, k):
        sid = self.ids[k]
        im = Image.open(IMAGES / f'{sid}.png').convert('L')
        w = min(2400, max(8, round(im.width * HEIGHT / im.height)))  # cap stray wide crops
        im = im.resize((w, HEIGHT))
        x = torch.from_numpy(np.asarray(im, np.float32) / 255.0)[None]   # 1 x H x W
        txt = _txt(sid)
        y = torch.tensor([STOI[c] for c in txt if c in STOI], dtype=torch.long)
        return x, y, txt, sid

def collate(b):
    b = [r for r in b if len(r[1]) > 0]        # drop lines with no in-charset chars
    if not b:
        return None
    W = max(r[0].shape[2] for r in b)
    X = torch.zeros(len(b), 1, HEIGHT, W)
    for i, r in enumerate(b):
        X[i, :, :, :r[0].shape[2]] = r[0]
    ys = [r[1] for r in b]
    yl = torch.tensor([len(y) for y in ys], dtype=torch.long)
    return X, torch.cat(ys), yl, [r[2] for r in b], [r[3] for r in b]

class CRNN(nn.Module):
    def __init__(self, n_cls):
        super().__init__()
        def blk(i, o, pool):
            L = [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(True)]
            return L + ([nn.MaxPool2d(pool)] if pool else [])
        # width is halved twice (/4 total); height is collapsed by mean() in forward,
        # so this works for HEIGHT=64 or 48 without any kernel-size arithmetic.
        self.cnn = nn.Sequential(
            *blk(1, 64, (2, 2)), *blk(64, 128, (2, 2)),
            *blk(128, 256, None), *blk(256, 256, (2, 1)),
            *blk(256, 512, None), *blk(512, 512, (2, 1)))
        self.rnn = nn.LSTM(512, 256, 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, n_cls)
    def forward(self, x):
        f = self.cnn(x)                     # B x 512 x H' x W'
        f = f.mean(2).permute(0, 2, 1)       # B x W' x 512  (collapse height)
        f, _ = self.rnn(f)
        return self.fc(f).log_softmax(2)     # B x W' x n_cls

net = CRNN(len(CHARSET)).to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=LR)
steps_per_epoch = max(1, len(SPLIT['train']) // BATCH)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS * steps_per_epoch)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
start_step = _st.get('step', 0) if _st else 0
best_cer = _st.get('best_cer', 9.9) if _st else 9.9
bad = _st.get('bad', 0) if _st else 0
if _st is not None:
    net.load_state_dict(_st['model'])
    if 'opt' in _st: opt.load_state_dict(_st['opt'])
    if 'sched' in _st: sched.load_state_dict(_st['sched'])

def _decode(logp):
    # greedy CTC: argmax, collapse repeats, drop blanks (index 0)
    out = []
    for r in logp.argmax(2).cpu().numpy():
        prev, s = 0, []
        for t in r:
            if t != prev and t != 0: s.append(CHARSET[t])
            prev = t
        out.append(''.join(s))
    return out

@torch.no_grad()
def _eval(loader):
    net.eval(); num = {'raw': 0.0, 'light': 0.0, 'diac_insensitive': 0.0}; den = 0
    for batch in loader:
        if batch is None: continue
        X, _, _, ts, _ = batch
        for p, g in zip(_decode(net(X.to(dev))), ts):
            for pr in num: num[pr] += _cer(_norm(g, pr), _norm(p, pr))
            den += 1
    net.train()
    return {k: v / max(1, den) for k, v in num.items()}

def _save(step):
    tmp = CK.with_suffix('.tmp')
    torch.save({'model': net.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'step': step, 'best_cer': best_cer,
                'bad': bad, 'charset': CHARSET}, tmp)
    tmp.replace(CK)

## 7b. Train loop — resumes from the last checkpoint, saves every `SAVE_EVERY` steps

In [ ]:
if RUN in ('crnn', 'both'):
    tl = DataLoader(Lines(SPLIT['train']), BATCH, shuffle=True, num_workers=2,
                    collate_fn=collate, drop_last=True)
    vl = DataLoader(Lines(SPLIT['val']), BATCH, shuffle=False, num_workers=2, collate_fn=collate)
    step = start_step
    for epoch in range(start_step // steps_per_epoch, MAX_EPOCHS):
        for batch in tl:
            if batch is None: continue
            X, Y, yl, ts, _ = batch
            lp = net(X.to(dev)).permute(1, 0, 2)                 # T x B x C
            inl = torch.full((X.size(0),), lp.size(0), dtype=torch.long)  # full width; padded => blanks
            loss = ctc(lp, Y.to(dev), inl, yl)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            opt.step(); sched.step(); step += 1
            if step % SAVE_EVERY == 0:
                _save(step)
                print(f'  step {step}  loss {loss.item():.3f}  (ckpt -> Drive)', flush=True)
        if epoch % EVAL_EVERY == 0:
            m = _eval(vl)
            with open(MET, 'a') as fh:
                fh.write(json.dumps({'epoch': epoch, 'step': step, 'val_cer': m, 'ts': time.time()}) + '\n')
            print(f"epoch {epoch}  val CER L1/L2/L3 = "
                  f"{m['raw']:.4f}/{m['light']:.4f}/{m['diac_insensitive']:.4f}", flush=True)
            if m['diac_insensitive'] < best_cer - 1e-4:
                best_cer, bad = m['diac_insensitive'], 0
                torch.save({'model': net.state_dict(), 'charset': CHARSET, 'val_cer': m},
                           WORK / 'crnn' / 'best.pt')
            else:
                bad += 1
            _save(step)
            if bad >= PATIENCE:
                print('early stop — no val-CER improvement for', PATIENCE, 'evals'); break
    print('CRNN training done. best val CER(L3) =', round(best_cer, 4))

## 7c. CRNN test — CER at L1/L2/L3 + per-script

In [ ]:
if RUN in ('crnn', 'both'):
    from collections import defaultdict
    bp = WORK / 'crnn' / 'best.pt'
    if bp.exists():
        net.load_state_dict(torch.load(bp, map_location=dev)['model'])
    net.eval()
    agg = defaultdict(lambda: {'raw': 0.0, 'light': 0.0, 'diac_insensitive': 0.0, 'n': 0})
    dl = DataLoader(Lines(SPLIT['test']), BATCH, shuffle=False, num_workers=2, collate_fn=collate)
    with torch.no_grad():
        for batch in dl:
            if batch is None: continue
            X, _, _, ts, sids = batch
            for p, g, sid in zip(_decode(net(X.to(dev))), ts, sids):
                for grp in ('all', SCRIPT.get(sid) or 'unknown'):
                    for pr in ('raw', 'light', 'diac_insensitive'):
                        agg[grp][pr] += _cer(_norm(g, pr), _norm(p, pr))
                    agg[grp]['n'] += 1
    res = {g: {**{pr: round(v[pr] / max(1, v['n']), 4) for pr in ('raw', 'light', 'diac_insensitive')},
               'n': v['n']} for g, v in agg.items()}
    print(json.dumps(res, indent=2, ensure_ascii=False))
    print('RefLAM Kraken: weighted 23.31 | maghrebi 32.71 | naskh 22.62 | ruqah 11.65')
    (WORK / 'crnn' / 'result.json').write_text(json.dumps(res, indent=2, ensure_ascii=False))

## 8. Commit the results JSON back to the repo

Only the small `result.json` files — never the checkpoints.

In [ ]:
import shutil, subprocess, time, pathlib
rid = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
dst = pathlib.Path('results') / f'ctc-baseline-{rid}'
dst.mkdir(parents=True, exist_ok=True)
for p in (WORK / 'kraken' / 'result.json', WORK / 'crnn' / 'result.json', WORK / 'crnn' / 'metrics.jsonl'):
    if p.exists(): shutil.copy2(p, dst / p.name)
if os.environ.get('GH_TOKEN'):
    subprocess.run(['git', 'config', 'user.email', 'colab@arabic-ocr-bench'])
    subprocess.run(['git', 'config', 'user.name', 'colab:ctc-baseline'])
    subprocess.run(['git', 'add', '-f', str(dst)])
    subprocess.run(['git', 'commit', '-m', f'Tier-0 0.1 CTC baseline results ({rid})'])
    subprocess.run(['git', 'pull', '--rebase', '--autostash', 'origin', 'main'])
    print(subprocess.run(['git', 'push', 'origin', 'HEAD:main'], capture_output=True, text=True).stderr[-300:])
else:
    print('no GH_TOKEN — download', dst, 'from the file browser')

## Resume / notes

- **Disconnected?** Reopen, `Runtime → Run all`. Cell 6 warm-starts Kraken from
  its newest `arams_ft_*.mlmodel` on Drive; cell 7 loads `crnn/ckpt_last.pt`
  and continues from that step. Nothing before the last `SAVE_EVERY` step (or
  the last epoch, for Kraken) is lost.
- **`metrics.jsonl`** on Drive is the per-epoch val-CER trace — plot it or
  recover the best epoch even if the notebook never reached the final cell.
- If the CRNN CER is far from the RefLAM anchors, check the charset size /
  OOV print in cell 7 and the `HEIGHT` setting first — those, not the model,
  are the usual culprits.
- `RUN='crnn'` is the default (self-contained). Switch to `'both'` for the
  Kraken baseline once the CRNN trains; Kraken's CLI flags vary by version,
  so read cell 6's output if `ketos train` errors.
- The released AraMS `.gt.txt` is already the *normalised* transcription, so
  L1≈L2≈L3 here will be closer than on raw-diacritised corpora.